# MOFA-FLEX BP - Timing (3 runs)

💡 **Environment:** `clamp-analyses`  

In [1]:
import time
import numpy as np
import pandas as pd
import anndata as ad
import mofaflex as mfl
from pyprojroot import here
from pathlib import Path
import pickle

/home/msubirana/miniconda3/envs/clamp-analyses/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [5]:
K = pd.read_csv(here("output/gtex/CLAMP_K_gtex.csv"))

# Check which column exists and use it
if 'x' in K.columns:
    n_components = int(K['x'].iloc[0])
elif 'CLAMP_K_gtex' in K.columns:
    n_components = int(K['CLAMP_K_gtex'].iloc[0])
else:
    # Fallback: use the second column (index 1)
    n_components = int(K.iloc[0, 1])

print(f"Number of components: {n_components}")

Number of components: 412


In [6]:
gtex_data = pd.read_csv(here("output/gtex/df_gtex_fbm_filt.csv"), index_col=0).astype(np.float32)

output_dir = Path(here("output/model_performance/gtex"))
output_dir.mkdir(parents=True, exist_ok=True)

N_RUNS = 3

In [7]:
gene_list = gtex_data.index.tolist()

bp_collection = mfl.tl.msigdb_get_features(
    category="c5.go.bp",
    dbver="7.5.1"
)

bp_collection = bp_collection.filter(
    gene_list,
    min_fraction=0.4,
    min_count=40,
    max_count=200
)

bp_collection = bp_collection.merge_similar(
    metric="jaccard",
    similarity_threshold=0.8,
    iteratively=True,
)

INFO	Found 108 pairs to merge.
INFO	Found 4 pairs to merge.
INFO	Found 0 pairs to merge. Stopping...


In [8]:
adata = ad.AnnData(
    X=gtex_data.T.values,
    obs=pd.DataFrame(index=gtex_data.columns),
    var=pd.DataFrame(index=gtex_data.index)
)
adata.varm["annotations"] = bp_collection.to_mask(gene_list).T

In [9]:
MOFA_times = []

for i in range(N_RUNS):
    print(f"MOFA-FLEX BP run {i+1} of {N_RUNS}")
    
    start_time = time.time()
    
    data_options = mfl.DataOptions(
        scale_per_group=False,
        plot_data_overview=False,
        annotations_varm_key="annotations",
    )
    
    model_options = mfl.ModelOptions(
        n_factors=n_components,
        weight_prior="Horseshoe",
        likelihoods="Normal",
    )
    
    training_options = mfl.TrainingOptions(
        seed=42,
        max_epochs=1000,
        save_path=False,
        device="cpu",
    )
    
    model = mfl.MOFAFLEX(
        {"group_1": {"view_1": adata}},
        data_options,
        model_options,
        training_options,
    )
    
    end_time = time.time()
    elapsed = (end_time - start_time) / 60
    MOFA_times.append(elapsed)
    print(f"Run {i+1} time: {elapsed} minutes\n")

WARNING	Could not import dask. Data arrays may be copied, resulting in high memory usage.


MOFA-FLEX BP run 1 of 3


INFO	Initializing factors using `random` method...
100%|██████████| 1000/1000 [8:38:21<00:00, 31.10s/epochs, Loss=8.92e+4] 


Run 1 time: 533.1914184014003 minutes

MOFA-FLEX BP run 2 of 3


INFO	Initializing factors using `random` method...
100%|██████████| 1000/1000 [8:31:02<00:00, 30.66s/epochs, Loss=8.92e+4] 


Run 2 time: 526.1577870885532 minutes

MOFA-FLEX BP run 3 of 3


INFO	Initializing factors using `random` method...
100%|██████████| 1000/1000 [9:32:52<00:00, 34.37s/epochs, Loss=8.92e+4] 


Run 3 time: 590.3642619291942 minutes



In [10]:
MOFA_FLEX_BP_time_minutes = {f"run{i+1}": t for i, t in enumerate(MOFA_times)}
with open(output_dir / "MOFA_FLEX_BP_time_minutes.pkl", "wb") as f:
    pickle.dump(MOFA_FLEX_BP_time_minutes, f)
print(f"MOFA-FLEX BP times: {MOFA_FLEX_BP_time_minutes}")

MOFA-FLEX BP times: {'run1': 533.1914184014003, 'run2': 526.1577870885532, 'run3': 590.3642619291942}
